# 🎃 MLP Classification untuk Pumpkin Dataset

**Klasifikasi Biji Labu menggunakan Multi-Layer Perceptron (MLP)**

Notebook ini mencakup:
1. Load Data
2. Data Preprocessing
3. Eksplorasi Data & Visualisasi
4. Membangun Model MLP
5. Training Model
6. Evaluasi Model
7. Visualisasi Hasil

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

# Set style untuk plot
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

print("✓ Libraries imported successfully!")

## 2. Load Data

In [ ]:
# Load dataset
data = pd.read_csv('pumpkin.csv')

print(f"Shape dataset: {data.shape}")
print(f"Jumlah sampel: {data.shape[0]}")
print(f"Jumlah fitur: {data.shape[1] - 1}")

In [ ]:
# Tampilkan 5 baris pertama
data.head()

In [ ]:
# Informasi dataset
data.info()

In [ ]:
# Statistik deskriptif
data.describe()

## 3. Data Preprocessing

In [ ]:
# Cek nilai kosong (missing values)
print("Missing values per kolom:")
print(data.isnull().sum())
print(f"\nTotal missing values: {data.isnull().sum().sum()}")

In [ ]:
# Cek duplikasi
print(f"Jumlah duplikat: {data.duplicated().sum()}")

In [ ]:
# Distribusi kelas target
print("Distribusi kelas target:")
print(data['Class'].value_counts())
print(f"\nPersentase:")
print(data['Class'].value_counts(normalize=True) * 100)

In [ ]:
# Pisahkan fitur dan target
X = data.drop('Class', axis=1)
y = data['Class']

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"\nFitur: {X.columns.tolist()}")

In [ ]:
# Encode label target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Kelas asli: {label_encoder.classes_}")
print(f"Kelas encoded: {np.unique(y_encoded)}")

In [ ]:
# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Ukuran data training: {X_train.shape}")
print(f"Ukuran data testing: {X_test.shape}")
print(f"\nDistribusi kelas di training: {np.bincount(y_train)}")
print(f"Distribusi kelas di testing: {np.bincount(y_test)}")

In [ ]:
# Normalisasi/Standardisasi fitur menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data setelah StandardScaler (mean≈0, std≈1):")
print(f"Mean X_train_scaled: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Std X_train_scaled: {X_train_scaled.std(axis=0).round(4)}")

## 4. Eksplorasi Data & Visualisasi (Pre-Training)

### 4.1 Distribusi Kelas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

class_counts = data['Class'].value_counts()
colors = ['#3498db', '#e74c3c']

bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Kelas', fontsize=14)
ax.set_ylabel('Jumlah Sampel', fontsize=14)
ax.set_title('Distribusi Kelas Target (Pumpkin Dataset)', fontsize=16, fontweight='bold')

for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
            str(count), ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.2 Heatmap Korelasi

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

correlation_matrix = X.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 9}, cbar_kws={'shrink': 0.8})
ax.set_title('Matriks Korelasi Fitur', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.3 Distribusi Fitur per Kelas

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()
feature_names = X.columns.tolist()

for idx, feature in enumerate(feature_names):
    ax = axes[idx]
    for class_name, color in zip(label_encoder.classes_, colors):
        class_data = data[data['Class'] == class_name][feature]
        ax.hist(class_data, bins=30, alpha=0.6, label=class_name, color=color, edgecolor='white')
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Frekuensi', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_title(f'Distribusi {feature}', fontsize=11)

plt.suptitle('Distribusi Fitur per Kelas', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.4 Boxplot untuk Deteksi Outliers

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for idx, feature in enumerate(feature_names):
    ax = axes[idx]
    data.boxplot(column=feature, by='Class', ax=ax)
    ax.set_xlabel('Kelas', fontsize=10)
    ax.set_ylabel(feature, fontsize=10)
    ax.set_title(f'{feature}', fontsize=11)

plt.suptitle('Boxplot Fitur per Kelas', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.5 Pairplot Fitur Penting

In [ ]:
important_features = ['Area', 'Perimeter', 'Eccentricity', 'Compactness', 'Class']

fig = sns.pairplot(data[important_features], hue='Class', 
                   palette={'Çerçevelik': '#3498db', 'Ürgüp Sivrisi': '#e74c3c'},
                   diag_kind='hist', height=2.5, aspect=1)
fig.fig.suptitle('Pairplot Fitur Penting', fontsize=16, fontweight='bold', y=1.02)
plt.show()

## 5. Membangun dan Melatih Model MLP

In [ ]:
# Definisikan arsitektur MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),  # 3 hidden layers
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size='auto',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=1000,
    shuffle=True,
    random_state=42,
    tol=1e-4,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

print("="*60)
print("ARSITEKTUR MLP")
print("="*60)
print(f"Hidden Layers     : (100, 50, 25)")
print(f"Activation        : ReLU")
print(f"Solver            : Adam")
print(f"Learning Rate     : Adaptive (init=0.001)")
print(f"Max Iterations    : 1000")
print(f"Early Stopping    : Enabled")
print(f"Validation Frac   : 10%")
print("="*60)

In [ ]:
# Melatih model
print("\nMelatih model MLP...\n")
mlp.fit(X_train_scaled, y_train)

print(f"\n{'='*60}")
print("TRAINING SELESAI!")
print(f"{'='*60}")
print(f"Jumlah iterasi: {mlp.n_iter_}")
print(f"Skor validasi terbaik: {mlp.best_validation_score_:.4f}")

## 6. Visualisasi Loss Function

### 6.1 Training Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

iterations = range(1, len(mlp.loss_curve_) + 1)
ax.plot(iterations, mlp.loss_curve_, 'b-', linewidth=2, label='Training Loss')

ax.set_xlabel('Epoch/Iterasi', fontsize=14)
ax.set_ylabel('Loss', fontsize=14)
ax.set_title('Loss Function selama Training', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Anotasi minimum loss
min_loss = min(mlp.loss_curve_)
min_idx = mlp.loss_curve_.index(min_loss)
ax.annotate(f'Min Loss: {min_loss:.4f}\n(Epoch {min_idx+1})', 
            xy=(min_idx+1, min_loss), xytext=(min_idx+50, min_loss+0.02),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.show()

### 6.2 Validation Score Curve

In [ ]:
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_ is not None:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    iterations_val = range(1, len(mlp.validation_scores_) + 1)
    ax.plot(iterations_val, mlp.validation_scores_, 'g-', linewidth=2, label='Validation Score')
    
    ax.set_xlabel('Epoch/Iterasi', fontsize=14)
    ax.set_ylabel('Validation Score', fontsize=14)
    ax.set_title('Validation Score selama Training', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Anotasi max score
    max_val = max(mlp.validation_scores_)
    max_idx = mlp.validation_scores_.index(max_val)
    ax.annotate(f'Max Score: {max_val:.4f}\n(Epoch {max_idx+1})', 
                xy=(max_idx+1, max_val), xytext=(max_idx-30, max_val-0.03),
                arrowprops=dict(arrowstyle='->', color='darkgreen'),
                fontsize=10, color='darkgreen')
    
    plt.tight_layout()
    plt.show()
else:
    print("Validation scores tidak tersedia")

### 6.3 Training Loss vs Validation Score

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Loss curve
color1 = '#e74c3c'
ax1.plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_, 
         color=color1, linewidth=2, label='Training Loss')
ax1.set_xlabel('Epoch/Iterasi', fontsize=14)
ax1.set_ylabel('Training Loss', color=color1, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color1)

# Validation score di sumbu y kedua
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_ is not None:
    ax2 = ax1.twinx()
    color2 = '#27ae60'
    ax2.plot(range(1, len(mlp.validation_scores_) + 1), mlp.validation_scores_, 
             color=color2, linewidth=2, linestyle='--', label='Validation Score')
    ax2.set_ylabel('Validation Score', color=color2, fontsize=14)
    ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('Training Loss vs Validation Score', fontsize=16, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_ is not None:
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=12)
else:
    ax1.legend(fontsize=12)

plt.tight_layout()
plt.show()

## 7. Prediksi dan Evaluasi Model

In [ ]:
# Prediksi
y_pred = mlp.predict(X_test_scaled)
y_pred_proba = mlp.predict_proba(X_test_scaled)

print(f"Jumlah sampel test: {len(y_test)}")
print(f"Shape prediksi: {y_pred.shape}")
print(f"Shape probabilitas: {y_pred_proba.shape}")

In [ ]:
# Hitung metrik evaluasi
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print("="*60)
print("HASIL EVALUASI MODEL")
print("="*60)
print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("="*60)

In [ ]:
# Classification Report lengkap
print("\nClassification Report:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Cross-validation
print("Cross-Validation (5-fold):")
print("="*60)

cv_scores = cross_val_score(mlp, X_train_scaled, y_train, cv=5, scoring='accuracy')

print(f"CV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

## 8. Visualisasi Hasil Evaluasi

### 8.1 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(y_test, y_pred)

# Confusion Matrix (counts)
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
            annot_kws={'size': 20}, linewidths=0.5, linecolor='gray')
ax1.set_xlabel('Prediksi', fontsize=14)
ax1.set_ylabel('Aktual', fontsize=14)
ax1.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Confusion Matrix (normalized)
ax2 = axes[1]
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', ax=ax2,
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
            annot_kws={'size': 18}, linewidths=0.5, linecolor='gray')
ax2.set_xlabel('Prediksi', fontsize=14)
ax2.set_ylabel('Aktual', fontsize=14)
ax2.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print confusion matrix details
print(f"\nTrue Çerçevelik → Çerçevelik   (TN): {cm[0,0]}")
print(f"True Çerçevelik → Ürgüp Sivrisi (FP): {cm[0,1]}")
print(f"True Ürgüp Sivrisi → Çerçevelik (FN): {cm[1,0]}")
print(f"True Ürgüp Sivrisi → Ürgüp Sivrisi (TP): {cm[1,1]}")

### 8.2 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])
roc_auc = auc(fpr, tpr)

ax.plot(fpr, tpr, color='#2980b9', linewidth=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#95a5a6', linewidth=2, linestyle='--', label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.3, color='#3498db')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=14)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=14)
ax.set_title('ROC Curve (Receiver Operating Characteristic)', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)

# Optimal threshold
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
ax.scatter([fpr[optimal_idx]], [tpr[optimal_idx]], marker='o', s=100, c='red', zorder=5)
ax.annotate(f'Optimal Point\n({fpr[optimal_idx]:.2f}, {tpr[optimal_idx]:.2f})', 
            xy=(fpr[optimal_idx], tpr[optimal_idx]), xytext=(fpr[optimal_idx]+0.1, tpr[optimal_idx]-0.1),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')

plt.tight_layout()
plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")

### 8.3 Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_test, y_pred_proba[:, 1])
pr_auc = auc(recall_curve, precision_curve)

ax.plot(recall_curve, precision_curve, color='#8e44ad', linewidth=2.5, 
        label=f'PR Curve (AUC = {pr_auc:.4f})')
ax.fill_between(recall_curve, precision_curve, alpha=0.3, color='#9b59b6')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall', fontsize=14)
ax.set_ylabel('Precision', fontsize=14)
ax.set_title('Precision-Recall Curve', fontsize=16, fontweight='bold')
ax.legend(loc='lower left', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PR-AUC Score: {pr_auc:.4f}")

### 8.4 Performance Metrics Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metrics_values = [accuracy, precision, recall, f1, roc_auc]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

bars = ax.bar(metrics_names, metrics_values, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Score', fontsize=14)
ax.set_title('Model Performance Metrics', fontsize=16, fontweight='bold')
ax.set_ylim([0, 1.1])

# Tambahkan nilai di atas bar
for bar, val in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Garis threshold 90%
ax.axhline(y=0.9, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Threshold 90%')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 8.5 Learning Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

train_sizes, train_scores, test_scores = learning_curve(
    mlp, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

ax.fill_between(train_sizes, train_scores_mean - train_scores_std,
                train_scores_mean + train_scores_std, alpha=0.2, color='#3498db')
ax.fill_between(train_sizes, test_scores_mean - test_scores_std,
                test_scores_mean + test_scores_std, alpha=0.2, color='#e74c3c')
ax.plot(train_sizes, train_scores_mean, 'o-', color='#3498db', linewidth=2, label='Training Score')
ax.plot(train_sizes, test_scores_mean, 's-', color='#e74c3c', linewidth=2, label='Cross-Validation Score')

ax.set_xlabel('Training Set Size', fontsize=14)
ax.set_ylabel('Score', fontsize=14)
ax.set_title('Learning Curve', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.7, 1.02])

plt.tight_layout()
plt.show()

### 8.6 Feature Importance (Weight Magnitude)

In [ ]:
# Hitung feature importance berdasarkan magnitude weight layer pertama
weights_layer1 = np.abs(mlp.coefs_[0])
feature_importance = np.mean(weights_layer1, axis=1)
feature_importance_normalized = feature_importance / feature_importance.sum()

fig, ax = plt.subplots(figsize=(12, 8))

sorted_idx = np.argsort(feature_importance_normalized)
y_pos = np.arange(len(feature_names))

colors_gradient = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feature_names)))
ax.barh(y_pos, feature_importance_normalized[sorted_idx], align='center', 
        color=colors_gradient, edgecolor='black', linewidth=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=11)
ax.set_xlabel('Normalized Weight Magnitude', fontsize=14)
ax.set_title('Feature Importance (Based on MLP Weight Magnitude)', fontsize=16, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Tambahkan nilai
for i, v in enumerate(feature_importance_normalized[sorted_idx]):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 8.7 Cross-Validation Scores Boxplot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate CV dengan multiple metrics
cv_accuracy = cross_val_score(mlp, X_train_scaled, y_train, cv=5, scoring='accuracy')
cv_precision = cross_val_score(mlp, X_train_scaled, y_train, cv=5, scoring='precision')
cv_recall = cross_val_score(mlp, X_train_scaled, y_train, cv=5, scoring='recall')
cv_f1 = cross_val_score(mlp, X_train_scaled, y_train, cv=5, scoring='f1')

cv_data = [cv_accuracy, cv_precision, cv_recall, cv_f1]
bp = ax.boxplot(cv_data, labels=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                patch_artist=True, widths=0.6)

colors_box = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Score', fontsize=14)
ax.set_title('Cross-Validation Scores (5-Fold)', fontsize=16, fontweight='bold')
ax.set_ylim([0.7, 1.05])
ax.grid(axis='y', alpha=0.3)

# Tambahkan mean values
for i, (cv, color) in enumerate(zip(cv_data, colors_box)):
    ax.scatter([i+1], [np.mean(cv)], marker='D', s=100, color='white', edgecolor='black', zorder=5)
    ax.text(i+1.15, np.mean(cv), f'{np.mean(cv):.3f}', fontsize=10, va='center')

plt.tight_layout()
plt.show()

### 8.8 Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribusi prediksi vs aktual
ax1 = axes[0]
pred_counts = pd.Series(y_pred).value_counts().sort_index()
actual_counts = pd.Series(y_test).value_counts().sort_index()

x = np.arange(2)
width = 0.35

bars1 = ax1.bar(x - width/2, actual_counts, width, label='Actual', color='#3498db', edgecolor='black')
bars2 = ax1.bar(x + width/2, pred_counts, width, label='Predicted', color='#e74c3c', edgecolor='black')

ax1.set_xlabel('Class', fontsize=14)
ax1.set_ylabel('Count', fontsize=14)
ax1.set_title('Actual vs Predicted Distribution', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(label_encoder.classes_, fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Prediction probability distribution
ax2 = axes[1]
for i, class_name in enumerate(label_encoder.classes_):
    class_probs = y_pred_proba[y_test == i, i]
    ax2.hist(class_probs, bins=20, alpha=0.6, label=f'{class_name}', 
             color=colors[i], edgecolor='white')
    ax2.axvline(np.mean(class_probs), color=colors[i], linestyle='--', linewidth=2)

ax2.set_xlabel('Prediction Probability', fontsize=14)
ax2.set_ylabel('Frequency', fontsize=14)
ax2.set_title('Prediction Probability Distribution', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Ringkasan Akhir

In [ ]:
print("="*70)
print("          RINGKASAN MLP CLASSIFICATION - PUMPKIN DATASET")
print("="*70)
print()
print("📊 DATASET INFO")
print(f"   • Total Samples     : {data.shape[0]}")
print(f"   • Total Features    : {data.shape[1]-1}")
print(f"   • Classes           : {', '.join(label_encoder.classes_)}")
print(f"   • Train Size        : {X_train.shape[0]}")
print(f"   • Test Size         : {X_test.shape[0]}")
print()
print("🏗️ MODEL ARCHITECTURE")
print(f"   • Hidden Layers     : (100, 50, 25)")
print(f"   • Activation        : ReLU")
print(f"   • Solver            : Adam")
print(f"   • Iterations        : {mlp.n_iter_}")
print()
print("📈 PERFORMANCE METRICS (Test Set)")
print(f"   • Accuracy          : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   • Precision         : {precision:.4f}")
print(f"   • Recall            : {recall:.4f}")
print(f"   • F1-Score          : {f1:.4f}")
print(f"   • ROC-AUC           : {roc_auc:.4f}")
print(f"   • Mean CV Score     : {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print()
print("📋 CONFUSION MATRIX")
print(f"   • True Çerçevelik  → Çerçevelik   : {cm[0,0]}")
print(f"   • True Çerçevelik  → Ürgüp Sivrisi: {cm[0,1]}")
print(f"   • True Ürgüp Sivrisi → Çerçevelik  : {cm[1,0]}")
print(f"   • True Ürgüp Sivrisi → Ürgüp Sivrisi: {cm[1,1]}")
print()
print("="*70)
print("                     KLASIFIKASI SELESAI! ✅")
print("="*70)